In [ ]:
import os
import random
import re
from datetime import datetime, timedelta
from pathlib import Path

missing_modules = []

try:
    import numpy as np
except ModuleNotFoundError:
    np = None
    missing_modules.append("numpy")

try:
    import rasterio
    from rasterio.windows import Window
except ModuleNotFoundError:
    rasterio = None
    Window = None
    missing_modules.append("rasterio")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except ModuleNotFoundError:
    torch = None
    nn = None
    Dataset = None
    DataLoader = None
    missing_modules.append("torch")


# =========================
# 1. Basic Settings
# =========================

image_dir = "E:/satellite_data/02grassland_fire/image_6band_without_smoke"
label_dir = "E:/satellite_data/02grassland_fire/label_6band_without_smoke"

patch_size = 128
stride = 128

batch_size = 32
num_epochs = 20
learning_rate = 1e-4

min_positive_pixels = 10
negative_keep_ratio = 0.3
max_patches_per_image = 300

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15
lead_days = 3
threshold = 0.3

random_seed = 42
random.seed(random_seed)

if np is not None:
    np.random.seed(random_seed)

if torch is not None:
    torch.manual_seed(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = "cpu"

print("Device:", device)

if missing_modules:
    print("Missing modules:", ", ".join(missing_modules))
    print("Install first: pip install numpy rasterio torch")


# =========================
# 2. Match Image With 3-Day-Ahead Label
# =========================

TIME_PATTERN = re.compile(r"(\d{8}T\d{6})")
SPATIAL_KEY_PATTERN = re.compile(r"_(T\d{2}[A-Z]{3})_\d{8}T\d{6}_(R\d+_C\d+_\d+)$")


def extract_acquisition_time(path):
    matches = TIME_PATTERN.findall(path.stem)
    if not matches:
        return None
    return datetime.strptime(matches[0], "%Y%m%dT%H%M%S")


def extract_spatial_key(path):
    match = SPATIAL_KEY_PATTERN.search(path.stem)
    if match is None:
        return None
    return f"{match.group(1)}_{match.group(2)}"


image_dir = Path(image_dir)
label_dir = Path(label_dir)

print("image_dir:", image_dir)
print("label_dir:", label_dir)
print("lead_days:", lead_days)
print("threshold:", threshold)

pairs = []
lead_time_offsets = []

if image_dir.exists() and label_dir.exists():
    image_files = sorted(image_dir.glob("*.tif"))
    label_files = sorted(label_dir.glob("*.tif"))

    label_groups = {}
    skipped_label_files = 0

    for lab_path in label_files:
        lab_time = extract_acquisition_time(lab_path)
        spatial_key = extract_spatial_key(lab_path)

        if lab_time is None or spatial_key is None:
            skipped_label_files += 1
            continue

        label_groups.setdefault(spatial_key, []).append((lab_time, lab_path))

    for spatial_key in label_groups:
        label_groups[spatial_key].sort(key=lambda item: item[0])

    skipped_image_files = 0

    for img_path in image_files:
        img_time = extract_acquisition_time(img_path)
        spatial_key = extract_spatial_key(img_path)

        if img_time is None or spatial_key is None:
            skipped_image_files += 1
            continue

        target_time = img_time + timedelta(days=lead_days)
        candidates = label_groups.get(spatial_key, [])
        future_candidates = [
            (lab_time, lab_path)
            for lab_time, lab_path in candidates
            if lab_time >= target_time
        ]

        if future_candidates:
            matched_time, matched_label_path = min(future_candidates, key=lambda item: item[0])
            pairs.append((img_path, matched_label_path))
            lead_time_offsets.append((matched_time - img_time).days)

    print("skipped_image_files:", skipped_image_files)
    print("skipped_label_files:", skipped_label_files)
else:
    print("Image or label directory was not found.")

print("matched image-label pairs:", len(pairs))

if lead_time_offsets:
    print("actual lead days min/max:", min(lead_time_offsets), max(lead_time_offsets))
    print("example actual lead days:", sorted(set(lead_time_offsets))[:10])

if len(pairs) == 0:
    raise ValueError("No valid 3-day-ahead pairs were found. Please check the filename timestamps.")


# =========================
# 3. Split Train / Val / Test
# =========================

random.shuffle(pairs)

n_total = len(pairs)
n_train = int(n_total * train_ratio)
n_val = int(n_total * val_ratio)

train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train + n_val]
test_pairs = pairs[n_train + n_val:]

print("train_pairs:", len(train_pairs))
print("val_pairs:", len(val_pairs))
print("test_pairs:", len(test_pairs))


In [ ]:
# =========================
# 4. 根据 label 生成 patch 样本
# =========================

def build_samples(pair_list, set_name="train"):
    samples = []

    for img_path, lab_path in pair_list:
        with rasterio.open(lab_path) as lab_src:
            label = lab_src.read(1)

        height, width = label.shape

        coords = []

        for y in range(0, height - patch_size + 1, stride):
            for x in range(0, width - patch_size + 1, stride):
                lab_patch = label[y:y + patch_size, x:x + patch_size]

                positive_pixels = np.sum(lab_patch > 0)

                if positive_pixels >= min_positive_pixels:
                    cls = 1
                    coords.append((img_path, lab_path, x, y, cls))
                else:
                    if random.random() < negative_keep_ratio:
                        cls = 0
                        coords.append((img_path, lab_path, x, y, cls))

        if len(coords) > max_patches_per_image:
            coords = random.sample(coords, max_patches_per_image)

        samples.extend(coords)

    positive_num = sum(s[-1] == 1 for s in samples)
    negative_num = sum(s[-1] == 0 for s in samples)

    print(f"{set_name} patch 总数:", len(samples))
    print(f"{set_name} 正样本数:", positive_num)
    print(f"{set_name} 负样本数:", negative_num)

    return samples


train_samples = build_samples(train_pairs, "训练集")
val_samples = build_samples(val_pairs, "验证集")
test_samples = build_samples(test_pairs, "测试集")

if len(train_samples) == 0:
    raise ValueError("训练集没有生成任何 patch，请检查 label 或调小 patch_size。")


# =========================
# 5. 构建 PyTorch Dataset
# =========================

class FirePatchDataset(Dataset):
    def __init__(self, samples, use_indices=True):
        self.samples = samples
        self.use_indices = use_indices

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lab_path, x, y, cls = self.samples[idx]

        with rasterio.open(img_path) as src:
            img = src.read(window=Window(x, y, patch_size, patch_size))

        img = img.astype(np.float32)

        img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

        img = np.clip(img, 0, 10000) / 10000.0

        if self.use_indices:
            b2 = img[0]
            b3 = img[1]
            b4 = img[2]
            b8 = img[3]
            b11 = img[4]
            b12 = img[5]

            ndvi = (b8 - b4) / (b8 + b4 + 1e-6)
            nbr = (b8 - b12) / (b8 + b12 + 1e-6)

            ndvi = np.clip(ndvi, -1, 1)
            nbr = np.clip(nbr, -1, 1)

            img = np.concatenate(
                [
                    img,
                    ndvi[np.newaxis, :, :],
                    nbr[np.newaxis, :, :]
                ],
                axis=0
            )

        x_tensor = torch.tensor(img, dtype=torch.float32)
        y_tensor = torch.tensor(cls, dtype=torch.long)

        return x_tensor, y_tensor


use_indices = True

train_dataset = FirePatchDataset(train_samples, use_indices=use_indices)
val_dataset = FirePatchDataset(val_samples, use_indices=use_indices)
test_dataset = FirePatchDataset(test_samples, use_indices=use_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

input_channels = 8 if use_indices else 6

print("输入通道数:", input_channels)

In [ ]:
# =========================
# 6. 定义小 CNN 模型
# =========================

class SmallFireCNN(nn.Module):
    def __init__(self, in_channels=8, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SmallFireCNN(in_channels=input_channels, num_classes=2).to(device)


# =========================
# 7. 损失函数和优化器
# =========================

train_positive = sum(s[-1] == 1 for s in train_samples)
train_negative = sum(s[-1] == 0 for s in train_samples)

class_weights = torch.tensor(
    [
        1.0,
        train_negative / max(train_positive, 1)
    ],
    dtype=torch.float32
).to(device)

print("类别权重:", class_weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)


# =========================
# 8. 评价函数
# =========================

def evaluate(model, loader, threshold=0.5):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_num = 0

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = (probs >= threshold).long()

            total_loss += loss.item() * images.size(0)
            total_correct += (preds == labels).sum().item()
            total_num += images.size(0)

            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()
            tn += ((preds == 0) & (labels == 0)).sum().item()

    avg_loss = total_loss / max(total_num, 1)
    acc = total_correct / max(total_num, 1)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return avg_loss, acc, precision, recall, f1, tp, fp, fn, tn


In [ ]:
# =========================
# 9. 模型训练
# =========================

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

best_val_f1 = 0.0
best_model_path = "small_fire_cnn_best.pth"

for epoch in range(num_epochs):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_num = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        train_loss += loss.item() * images.size(0)
        train_correct += (preds == labels).sum().item()
        train_num += images.size(0)

    train_loss = train_loss / max(train_num, 1)
    train_acc = train_correct / max(train_num, 1)

    val_loss, val_acc, val_precision, val_recall, val_f1, tp, fp, fn, tn = evaluate(model, val_loader, threshold=threshold)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_precision"].append(val_precision)
    history["val_recall"].append(val_recall)
    history["val_f1"].append(val_f1)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f} "
        f"Val Precision: {val_precision:.4f} "
        f"Val Recall: {val_recall:.4f} "
        f"Val F1: {val_f1:.4f}"
    )

    print(f"验证集混淆矩阵: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print("保存当前最优模型")


# =========================
# 10. 测试集评估
# =========================

model.load_state_dict(torch.load(best_model_path, map_location=device))

test_loss, test_acc, test_precision, test_recall, test_f1, tp, fp, fn, tn = evaluate(model, test_loader, threshold=threshold)

print("\n测试集结果:")
print("Test Loss:", round(test_loss, 4))
print("Test Acc:", round(test_acc, 4))
print("Test Precision:", round(test_precision, 4))
print("Test Recall:", round(test_recall, 4))
print("Test F1:", round(test_f1, 4))

print("\n测试集混淆矩阵:")
print("TP:", tp, "FP:", fp)
print("FN:", fn, "TN:", tn)

In [ ]:
# =========================
# 11. Training Curves And Prediction Visualization
# =========================

import matplotlib.pyplot as plt

if "history" not in globals() or len(history["train_loss"]) == 0:
    raise ValueError("history is empty. Please run the training cell first.")

if len(test_pairs) == 0:
    raise ValueError("test_pairs is empty. Please run the data split cell first.")

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(epochs, history["train_loss"], marker="o", label="Train Loss")
axes[0, 0].plot(epochs, history["val_loss"], marker="s", label="Val Loss")
axes[0, 0].set_title("Loss Curve")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(epochs, history["train_acc"], marker="o", label="Train Acc")
axes[0, 1].plot(epochs, history["val_acc"], marker="s", label="Val Acc")
axes[0, 1].set_title("Accuracy Curve")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[0, 2].plot(epochs, history["val_precision"], marker="o", label="Precision")
axes[0, 2].plot(epochs, history["val_recall"], marker="s", label="Recall")
axes[0, 2].plot(epochs, history["val_f1"], marker="^", label="F1")
axes[0, 2].set_title("Validation Metrics")
axes[0, 2].set_xlabel("Epoch")
axes[0, 2].set_ylabel("Score")
axes[0, 2].set_ylim(0, 1.05)
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].legend()

sample_img_path, sample_lab_path = test_pairs[0]

model.eval()
with rasterio.open(sample_img_path) as src:
    image_data = src.read().astype(np.float32)

with rasterio.open(sample_lab_path) as src:
    label_data = src.read(1)

image_data = np.nan_to_num(image_data, nan=0.0, posinf=0.0, neginf=0.0)
image_data = np.clip(image_data, 0, 10000) / 10000.0

height, width = label_data.shape
prob_map = np.full((height, width), np.nan, dtype=np.float32)

for y in range(0, height - patch_size + 1, stride):
    for x in range(0, width - patch_size + 1, stride):
        patch = image_data[:, y:y + patch_size, x:x + patch_size]

        if use_indices:
            b4 = patch[2]
            b8 = patch[3]
            b12 = patch[5]
            ndvi = (b8 - b4) / (b8 + b4 + 1e-6)
            nbr = (b8 - b12) / (b8 + b12 + 1e-6)
            ndvi = np.clip(ndvi, -1, 1)
            nbr = np.clip(nbr, -1, 1)
            patch = np.concatenate([patch, ndvi[np.newaxis, :, :], nbr[np.newaxis, :, :]], axis=0)

        patch_tensor = torch.tensor(patch, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(patch_tensor)
            prob = torch.softmax(logits, dim=1)[0, 1].item()

        prob_map[y:y + patch_size, x:x + patch_size] = prob

rgb = image_data[[3, 2, 1]].transpose(1, 2, 0)
rgb = np.clip(rgb * 2.5, 0, 1)

axes[1, 0].imshow(rgb)
axes[1, 0].set_title("Test Image (RGB)")
axes[1, 0].axis("off")

axes[1, 1].imshow(label_data, cmap="gray")
axes[1, 1].set_title("Ground Truth")
axes[1, 1].axis("off")

im = axes[1, 2].imshow(prob_map, cmap="hot", vmin=0, vmax=1)
axes[1, 2].set_title(f"Predicted Fire Probability (th={threshold:.2f})")
axes[1, 2].axis("off")
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("Visualization image:", sample_img_path.name)
